# 00_CNN_PreProcessing_existing_234_mouseROI

기존 `02_CNN_train.ipynb`, `03_CNN_evaluation.ipynb`, `04_realtest_v4.ipynb`를 건드리지 않고 돌리기 위한 CNN 전처리 전용 노트북입니다.

- 입력 영상: `~/Desktop/energy/Dataset`
- 출력: `~/Desktop/CNN_code`
- 라벨: `0=정상`, `1=이상/볼링`, `2=검정화면`
- ROI는 마우스로 드래그해서 한 번 지정합니다.
- 기존 02/03 코드가 읽는 `numpyarray/*.npy`까지 생성합니다.

In [ ]:
from pathlib import Path

# =========================
# 경로 설정
# =========================
RAW_DATASET_DIR = Path.home() / "Desktop" / "energy" / "Dataset"
OUT_DIR = Path.home() / "Desktop" / "CNN_code"

# 기존 02/03/04 코드가 읽을 기본 구조
# OUT_DIR/
#   train/0, train/1, train/2
#   val/0,   val/1,   val/2
#   test/0,  test/1,  test/2
#   numpyarray/x_train.npy ...

# =========================
# 이미지 / 프레임 설정
# =========================
IMG_SIZE = 224

# 메모리 때문에 기본은 3프레임마다 1장 저장.
# 모든 프레임을 쓰려면 1로 바꾸면 됩니다.
FRAME_STRIDE = 3
START_FRAME = 0
MAX_FRAMES_PER_VIDEO = 0   # 0이면 전체 프레임 사용
JPG_QUALITY = 95

# 검정화면 판정 기준: crop grayscale 평균 밝기가 이 값 이하면 class 2
DARK_MEAN_THRESHOLD = 5.0

# 기존 02_CNN_train.ipynb가 numpyarray를 읽으므로 True 유지
MAKE_NUMPY = True

# 다시 돌릴 때 기존 train/val/test/numpyarray를 지울지 여부
CLEAN_OUTPUT = True

# =========================
# ROI 선택 설정
# =========================
# mouse: 첫 영상 프레임에서 마우스로 드래그하여 ROI 지정
# manual: 아래 MANUAL_ROI 값을 사용
# none: crop 없이 전체 frame 사용
ROI_MODE = "mouse"
FORCE_RESELECT_ROI = True
ROI_SELECT_FRAME_IDX = 50
ROI_SELECT_MAX_WIDTH = 1280
MANUAL_ROI = None  # 예: (120, 720, 250, 1130) = top,bottom,left,right

# =========================
# split 설정
# =========================
# Test* 폴더는 train/test로 video-level split
# Val* 폴더는 val로 사용
TEST_RATIO_FROM_TEST_FOLDERS = 0.2
RANDOM_SEED = 42

print("RAW_DATASET_DIR:", RAW_DATASET_DIR)
print("OUT_DIR        :", OUT_DIR)
print("FRAME_STRIDE   :", FRAME_STRIDE)
print("ROI_MODE       :", ROI_MODE)


In [ ]:
import os
import re
import cv2
import json
import time
import math
import shutil
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tensorflow.keras.utils import to_categorical

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
CLASSES = ["0", "1", "2"]

# =========================
# 라벨 정의
# 0 = 정상, 1 = 이상/볼링, 2 = 검정화면
# =========================
NORMAL_TEST_IDS = {1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 28, 29, 30, 44}
ABNORMAL_TEST_IDS = {6, 31, 45} | set(range(32, 44))
ABNORMAL_VAL_IDS = {1, 2}
NORMAL_VAL_IDS = set(range(3, 10))


def parse_folder_id(folder_name: str):
    m = re.match(r"^(Test|Val)(\d+)_", folder_name, flags=re.IGNORECASE)
    if not m:
        return None, None
    prefix = m.group(1).capitalize()
    idx = int(m.group(2))
    return prefix, idx


def base_label_from_folder(folder_name: str):
    prefix, idx = parse_folder_id(folder_name)
    if prefix == "Test":
        if idx in NORMAL_TEST_IDS:
            return 0
        if idx in ABNORMAL_TEST_IDS:
            return 1
    elif prefix == "Val":
        if idx in NORMAL_VAL_IDS:
            return 0
        if idx in ABNORMAL_VAL_IDS:
            return 1
    raise ValueError(f"라벨 매핑이 없는 폴더입니다: {folder_name}")


def find_video_folders(raw_dir: Path):
    rows = []
    for folder in sorted(raw_dir.iterdir(), key=lambda p: p.name):
        if not folder.is_dir() or folder.name.startswith("."):
            continue
        mp4s = [p for p in folder.glob("*.mp4") if not p.name.startswith("._")]
        if not mp4s:
            continue
        prefix, idx = parse_folder_id(folder.name)
        if prefix not in {"Test", "Val"}:
            continue
        try:
            label = base_label_from_folder(folder.name)
        except Exception as e:
            rows.append({
                "folder": folder.name,
                "path": str(folder),
                "video": mp4s[0].name,
                "prefix": prefix,
                "id": idx,
                "base_label": None,
                "error": str(e),
            })
            continue
        rows.append({
            "folder": folder.name,
            "path": str(folder),
            "video": mp4s[0].name,
            "video_path": str(mp4s[0]),
            "prefix": prefix,
            "id": idx,
            "base_label": int(label),
            "error": "",
        })
    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise RuntimeError(f"mp4 폴더를 찾지 못했습니다: {raw_dir}")
    return df


def make_video_level_split(df: pd.DataFrame, test_ratio: float, seed: int):
    df = df.copy()
    df["split"] = None

    # Val*는 validation으로 고정
    df.loc[df["prefix"] == "Val", "split"] = "val"

    rng = random.Random(seed)
    test_df = df[df["prefix"] == "Test"].copy()

    # 정상/이상 각각에서 test를 뽑아서 클래스 편향 방지
    for label in [0, 1]:
        idxs = test_df[test_df["base_label"] == label].index.tolist()
        rng.shuffle(idxs)
        n_test = max(1, int(round(len(idxs) * test_ratio))) if len(idxs) > 1 else 0
        test_idxs = set(idxs[:n_test])
        for i in idxs:
            df.loc[i, "split"] = "test" if i in test_idxs else "train"

    # 혹시 누락된 Test*는 train으로
    df.loc[df["split"].isna(), "split"] = "train"
    return df


def read_frame_at(video_path: Path, frame_idx: int):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"영상 열기 실패: {video_path}")
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    target_idx = min(max(0, frame_idx), max(0, frame_count - 1))
    cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)
    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        raise RuntimeError(f"프레임 읽기 실패: {video_path}, frame={target_idx}")
    return frame, target_idx


def select_roi_with_mouse(video_path: Path, frame_idx: int, max_width: int = 1280):
    frame, used_idx = read_frame_at(video_path, frame_idx)
    h, w = frame.shape[:2]
    scale = 1.0
    display_frame = frame.copy()
    if w > max_width:
        scale = max_width / float(w)
        display_frame = cv2.resize(display_frame, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

    window_name = "Drag ROI and press ENTER/SPACE. Cancel with C."
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    x, y, rw, rh = cv2.selectROI(window_name, display_frame, showCrosshair=True, fromCenter=False)
    cv2.destroyWindow(window_name)
    cv2.waitKey(1)

    if rw <= 0 or rh <= 0:
        raise RuntimeError("ROI 선택이 취소되었거나 잘못되었습니다.")

    left = int(round(x / scale))
    top = int(round(y / scale))
    right = int(round((x + rw) / scale))
    bottom = int(round((y + rh) / scale))

    left = max(0, min(left, w - 1))
    right = max(left + 1, min(right, w))
    top = max(0, min(top, h - 1))
    bottom = max(top + 1, min(bottom, h))

    return (top, bottom, left, right), frame, used_idx


def get_or_select_roi(df: pd.DataFrame, out_dir: Path):
    roi_json = out_dir / "roi_config.json"
    if ROI_MODE == "none":
        return None
    if ROI_MODE == "manual":
        if MANUAL_ROI is None:
            raise ValueError("ROI_MODE='manual'이면 MANUAL_ROI=(top,bottom,left,right)를 지정해야 합니다.")
        roi = tuple(map(int, MANUAL_ROI))
    elif ROI_MODE == "mouse":
        if roi_json.exists() and not FORCE_RESELECT_ROI:
            cfg = json.loads(roi_json.read_text(encoding="utf-8"))
            roi = tuple(cfg["roi"])
            print("기존 ROI 사용:", roi)
            return roi
        first_video = Path(df.dropna(subset=["video_path"]).iloc[0]["video_path"])
        print("ROI 선택 영상:", first_video)
        print("마우스로 crop할 영역을 드래그한 뒤 ENTER 또는 SPACE를 누르세요.")
        roi, frame, used_idx = select_roi_with_mouse(first_video, ROI_SELECT_FRAME_IDX, ROI_SELECT_MAX_WIDTH)
        print("선택 ROI:", roi)
        # preview 저장
        top, bottom, left, right = roi
        preview = frame.copy()
        cv2.rectangle(preview, (left, top), (right, bottom), (0, 0, 255), 3)
        cv2.imwrite(str(out_dir / "roi_preview.jpg"), preview)
        roi_json.write_text(json.dumps({
            "roi": roi,
            "source_video": str(first_video),
            "source_frame_idx": used_idx,
            "format": "top,bottom,left,right",
        }, ensure_ascii=False, indent=2), encoding="utf-8")
    else:
        raise ValueError(f"지원하지 않는 ROI_MODE: {ROI_MODE}")

    # coordinate.csv 생성: 기존 04 코드 coordinate_csv 모드도 쓸 수 있게 전체 행에 같은 ROI 저장
    save_coordinate_csv(out_dir, roi)
    return roi


def save_coordinate_csv(out_dir: Path, roi):
    top, bottom, left, right = roi
    # 기존 코드가 Test37이면 row index 36을 찾으므로 넉넉히 100행 생성
    coord = pd.DataFrame({
        "coordinate1": [f"{top}:{bottom}"] * 100,
        "coordinate2": [f"{left}:{right}"] * 100,
    })
    coord.to_csv(out_dir / "coordinate.csv", index=False, encoding="euc-kr")
    print("coordinate.csv 저장:", out_dir / "coordinate.csv")


def apply_roi(frame_bgr, roi):
    if roi is None:
        return frame_bgr
    top, bottom, left, right = roi
    return frame_bgr[top:bottom, left:right]


def is_dark_frame(gray_crop, threshold: float):
    if gray_crop.size == 0:
        return True
    return float(gray_crop.mean()) <= float(threshold)


def prepare_output_dirs(out_dir: Path, clean: bool):
    out_dir.mkdir(parents=True, exist_ok=True)
    for name in ["train", "val", "test", "numpyarray", "manifests"]:
        p = out_dir / name
        if clean and p.exists():
            shutil.rmtree(p)
    for split in ["train", "val", "test"]:
        for cls in CLASSES:
            (out_dir / split / cls).mkdir(parents=True, exist_ok=True)
    (out_dir / "manifests").mkdir(parents=True, exist_ok=True)


def process_videos(df: pd.DataFrame, roi, out_dir: Path):
    records = []
    counts = {"train": {"0": 0, "1": 0, "2": 0},
              "val": {"0": 0, "1": 0, "2": 0},
              "test": {"0": 0, "1": 0, "2": 0}}

    for _, row in df.iterrows():
        if row.get("error", ""):
            print("[SKIP]", row["folder"], row["error"])
            continue
        folder = row["folder"]
        video_path = Path(row["video_path"])
        split = row["split"]
        base_label = int(row["base_label"])

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            print("[경고] 영상 열기 실패:", video_path)
            continue

        fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        saved = 0
        dark_saved = 0
        frame_idx = 0
        t0 = time.time()

        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if MAX_FRAMES_PER_VIDEO and frame_idx >= MAX_FRAMES_PER_VIDEO:
                break
            if frame_idx < START_FRAME or ((frame_idx - START_FRAME) % FRAME_STRIDE != 0):
                frame_idx += 1
                continue

            crop = apply_roi(frame, roi)
            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            crop_mean = float(gray.mean()) if gray.size else 0.0

            label = 2 if is_dark_frame(gray, DARK_MEAN_THRESHOLD) else base_label
            if label == 2:
                dark_saved += 1

            resized = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            fname = f"{folder}_f{frame_idx:06d}.jpg"
            dst = out_dir / split / str(label) / fname
            cv2.imwrite(str(dst), resized, [int(cv2.IMWRITE_JPEG_QUALITY), int(JPG_QUALITY)])
            counts[split][str(label)] += 1
            saved += 1

            records.append({
                "img": fname,
                "path": str(dst),
                "folder": folder,
                "video": video_path.name,
                "frame_idx": frame_idx,
                "time_sec": frame_idx / fps if fps > 0 else np.nan,
                "split": split,
                "base_label": base_label,
                "label": int(label),
                "crop_mean": crop_mean,
                "is_dark": int(label == 2),
            })
            frame_idx += 1

        cap.release()
        print(f"{folder}: split={split}, base={base_label}, frames={frame_count}, saved={saved}, dark={dark_saved}, time={time.time()-t0:.1f}s")

    manifest = pd.DataFrame(records)
    manifest.to_csv(out_dir / "manifests" / "cnn_frame_manifest.csv", index=False, encoding="utf-8-sig")
    print("\n저장 개수:")
    print(pd.DataFrame(counts).T)
    return manifest, counts


def make_dataset_npy(dataset_dir: Path, categories, imgsize, one_hot=True, return_names=False):
    st = time.time()
    x_list = []
    y_list = []
    name_list = []

    for label, cls in enumerate(categories):
        img_dir = dataset_dir / cls
        if not img_dir.exists():
            print(f"[경고] 폴더 없음: {img_dir}")
            continue
        files = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])
        for img_path in files:
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"[경고] 이미지 읽기 실패: {img_path}")
                continue
            img = cv2.resize(img, dsize=(imgsize, imgsize), interpolation=cv2.INTER_AREA)
            img = img / 255.0
            x_list.append(img)
            y_list.append(label)
            if return_names:
                name_list.append(img_path.name)

    X = np.array(x_list, dtype=np.float32).reshape(-1, imgsize, imgsize, 1)
    y = np.array(y_list, dtype=np.int32)
    if one_hot:
        y = to_categorical(y, num_classes=len(categories))

    ed = time.time()
    print(f"{dataset_dir.name}: X {X.shape}, y {y.shape}, Time {ed-st:.1f}s")
    if return_names:
        return X, y, np.array(name_list)
    return X, y


def save_numpy_arrays(out_dir: Path):
    save_dir = out_dir / "numpyarray"
    save_dir.mkdir(exist_ok=True)

    categories = ["0", "1", "2"]
    x_train, y_train = make_dataset_npy(out_dir / "train", categories, IMG_SIZE, one_hot=True, return_names=False)
    x_val, y_val = make_dataset_npy(out_dir / "val", categories, IMG_SIZE, one_hot=True, return_names=False)
    x_test, y_test, x_test_name = make_dataset_npy(out_dir / "test", categories, IMG_SIZE, one_hot=True, return_names=True)

    np.save(save_dir / "x_train.npy", x_train)
    np.save(save_dir / "y_train.npy", y_train)
    np.save(save_dir / "x_val.npy", x_val)
    np.save(save_dir / "y_val.npy", y_val)
    np.save(save_dir / "x_test.npy", x_test)
    np.save(save_dir / "y_test.npy", y_test)
    np.save(save_dir / "x_test_name.npy", x_test_name)

    print("\n기존 02/03 코드용 numpyarray 저장 완료:", save_dir)
    print("x_train:", x_train.shape, "y_train:", y_train.shape)
    print("x_val  :", x_val.shape, "y_val  :", y_val.shape)
    print("x_test :", x_test.shape, "y_test :", y_test.shape)


In [ ]:
# 1) 폴더 스캔 + 라벨/split 생성
prepare_output_dirs(OUT_DIR, CLEAN_OUTPUT)

folder_df = find_video_folders(RAW_DATASET_DIR)
if folder_df["error"].fillna("").astype(bool).any():
    display(folder_df[folder_df["error"].fillna("").astype(bool)])
    raise RuntimeError("라벨 매핑이 없는 폴더가 있습니다. 위 표를 확인하세요.")

folder_df = make_video_level_split(folder_df, TEST_RATIO_FROM_TEST_FOLDERS, RANDOM_SEED)
folder_df.to_csv(OUT_DIR / "manifests" / "folder_split.csv", index=False, encoding="utf-8-sig")

print("폴더 split/label 요약")
display(folder_df[["folder", "prefix", "id", "base_label", "split", "video"]].sort_values(["prefix", "id"]))
print(folder_df.groupby(["split", "base_label"]).size())

# 2) 마우스로 ROI 선택
roi = get_or_select_roi(folder_df, OUT_DIR)
print("최종 ROI:", roi)

# 3) frame 추출 + crop + 0/1/2 저장
manifest, counts = process_videos(folder_df, roi, OUT_DIR)

# 4) 기존 02/03 코드가 바로 읽는 numpyarray 생성
if MAKE_NUMPY:
    save_numpy_arrays(OUT_DIR)

# 5) 요약 저장
summary = {
    "raw_dataset_dir": str(RAW_DATASET_DIR),
    "out_dir": str(OUT_DIR),
    "img_size": IMG_SIZE,
    "frame_stride": FRAME_STRIDE,
    "start_frame": START_FRAME,
    "max_frames_per_video": MAX_FRAMES_PER_VIDEO,
    "dark_mean_threshold": DARK_MEAN_THRESHOLD,
    "roi": roi,
    "n_frames_saved": int(len(manifest)),
    "counts": counts,
    "label_meaning": {"0": "normal", "1": "abnormal_balling", "2": "black_or_disconnect"},
}
with open(OUT_DIR / "manifests" / "preprocess_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n완료")
print("기존 02_CNN_train.ipynb 실행 위치:", OUT_DIR)
print("필수 파일:")
for p in [OUT_DIR / "numpyarray" / "x_train.npy", OUT_DIR / "numpyarray" / "y_train.npy", OUT_DIR / "numpyarray" / "x_val.npy", OUT_DIR / "numpyarray" / "y_val.npy", OUT_DIR / "numpyarray" / "x_test.npy", OUT_DIR / "numpyarray" / "y_test.npy", OUT_DIR / "numpyarray" / "x_test_name.npy"]:
    print(" -", p, "OK" if p.exists() else "MISSING")
